# UD5.01. Tensores y capas

**Módulo 5073 · Programación de Inteligencia Artificial · Curso 2026/27**
Bloques 1 y 2 de los apuntes · Criterio de evaluación **2.b**

---

Este cuaderno establece el vocabulario de toda la unidad. No entrena nada: construye
tensores, los mira, y comprueba que una capa densa es un producto de matrices y una suma.

Al terminar tienes que saber contestar sin ejecutar nada:

1. Qué forma tiene un lote de 64 imágenes en color de 96×96.
2. Cuántos parámetros tiene una capa densa de 32 unidades que recibe 100 entradas.
3. Por qué `modelo.predict(imagen)` falla y `modelo.predict(imagen[np.newaxis])` no.

In [ ]:
import numpy as np
import tensorflow as tf
import keras

# Reproducibilidad: fija de una vez Python, NumPy y TensorFlow.
keras.utils.set_random_seed(20262027)

print("tensorflow", tf.__version__)
print("keras      ", keras.__version__)
print("numpy      ", np.__version__)
print()
print("Dispositivos:", [d.device_type for d in tf.config.list_physical_devices()])

Si en la lista de dispositivos solo aparece `CPU`, estás en la máquina del aula y todo lo
de esta unidad funciona igual, solo que más despacio en la parte de imágenes. En Colab con
acelerador aparece también `GPU`.

---

## 1. Un tensor es un array con dos añadidos

Lo primero es comprobar que un tensor y un `ndarray` son la misma cosa vista desde dos
sitios, y que se pasa de uno a otro sin copiar nada raro.

In [ ]:
a = np.arange(6, dtype="float32").reshape(2, 3)
t = tf.constant(a)

print("ndarray:", type(a).__name__, a.shape, a.dtype)
print("tensor: ", type(t).__name__, tuple(t.shape), t.dtype.name)
print()
print("El tensor lleva encima donde vive, y el ndarray no:")
print("  ", t.device)
print()
print("Y se vuelve atras con .numpy():")
print("  ", type(t.numpy()).__name__, np.array_equal(a, t.numpy()))

### El segundo añadido: recuerda de dónde viene

Esta es la diferencia que importa, y la que hace posible entrenar. Dentro de un contexto
`GradientTape`, TensorFlow anota cada operación en una cinta y después la recorre hacia
atrás para calcular derivadas. Con un `ndarray` no hay cinta y no hay derivadas.

Comprobémoslo con una función cuya derivada sabemos: si $y = x^2$, entonces $dy/dx = 2x$,
que en $x=3$ vale 6.

In [ ]:
x = tf.Variable(3.0)

with tf.GradientTape() as cinta:
    y = x ** 2

print("y  =", y.numpy())
print("dy/dx =", cinta.gradient(y, x).numpy(), " (deberia ser 2*3 = 6)")

In [ ]:
# Una funcion con dos variables, para ver que devuelve el gradiente completo.
#   f(w, b) = (w * 2 + b - 7)^2      es el error cuadratico de una neurona
#                                     de una sola entrada, con x = 2 e y = 7
w = tf.Variable(1.0)
b = tf.Variable(0.0)

with tf.GradientTape() as cinta:
    prediccion = w * 2.0 + b
    perdida = (prediccion - 7.0) ** 2

dw, db = cinta.gradient(perdida, [w, b])
print(f"prediccion {prediccion.numpy():.1f}  perdida {perdida.numpy():.1f}")
print(f"dL/dw = {dw.numpy():.1f}    dL/db = {db.numpy():.1f}")
print()
print("A mano:  dL/dw = 2*(pred - 7)*2 =", 2 * (prediccion.numpy() - 7) * 2)
print("         dL/db = 2*(pred - 7)   =", 2 * (prediccion.numpy() - 7))

Eso de ahí arriba **es** el entrenamiento. Todo lo que viene después es lo mismo con más
parámetros, y una biblioteca que lo organiza.

---

## 2. El eje 0 cuenta muestras. Siempre.

La convención que hay que interiorizar antes que ninguna otra, porque de romperla salen
todos los errores de forma de la unidad.

In [ ]:
ejemplos = {
    "un escalar (una perdida)":            np.float32(0.42),
    "un cliente, 10 caracteristicas":      np.zeros(10, dtype="float32"),
    "300 clientes x 10 caracteristicas":   np.zeros((300, 10), dtype="float32"),
    "128 imagenes en gris de 28x28":       np.zeros((128, 28, 28), dtype="float32"),
    "128 imagenes en color de 224x224":    np.zeros((128, 224, 224, 3), dtype="float32"),
    "32 videos de 16 fotogramas":          np.zeros((32, 16, 224, 224, 3), dtype="float32"),
}

print(f"{'que es':38} {'forma':26} {'orden':>6} {'elementos':>12}")
print("-" * 86)
for nombre, t in ejemplos.items():
    print(f"{nombre:38} {str(np.shape(t)):26} {np.ndim(t):>6} {np.size(t):>12,}")

### El error de forma más frecuente de la unidad

Un modelo espera un **lote**. Una imagen suelta no lo es, y hay que convertirla en un lote
de una imagen. Vamos a provocar el error a propósito para reconocerlo cuando aparezca.

In [ ]:
# Un modelo minimo, solo para provocar el error.
modelo = keras.Sequential([
    keras.layers.Input(shape=(28, 28)),
    keras.layers.Flatten(),
    keras.layers.Dense(10, activation="softmax"),
])

imagen = np.zeros((28, 28), dtype="float32")

try:
    modelo.predict(imagen, verbose=0)
except Exception as e:
    print("FALLA, y asi es como se ve:")
    print(" ", str(e).split("\n")[0][:200])

print()
lote = imagen[np.newaxis]          # (28, 28) -> (1, 28, 28)
print("Con el eje de muestras:", lote.shape, "->",
      modelo.predict(lote, verbose=0).shape)

Hay tres formas de añadir ese eje y conviene conocer las tres, porque las tres aparecen en
el código que se encuentra por ahí:

| Forma | Comentario |
|---|---|
| `x[np.newaxis]` | la más legible. `np.newaxis` es literalmente `None` |
| `x[None]` | la misma, escrita corto |
| `np.expand_dims(x, axis=0)` | la explícita; obligatoria cuando el eje no es el 0 |

Y para el canal de una imagen en gris, el eje que se añade es el **último**:
`X[..., np.newaxis]`.

In [ ]:
imagen = np.zeros((28, 28), dtype="float32")

print("x[np.newaxis]          ", imagen[np.newaxis].shape)
print("x[None]                ", imagen[None].shape)
print("np.expand_dims(x, 0)   ", np.expand_dims(imagen, 0).shape)
print()
print("El canal va al final:")
print("x[..., np.newaxis]     ", imagen[..., np.newaxis].shape)
print("np.expand_dims(x, -1)  ", np.expand_dims(imagen, -1).shape)

---

## 3. El vocabulario del entrenamiento

Muestra, lote, época y paso. La cifra que importa **no es el número de épocas**: es el
número de pasos, que es el número de veces que se actualizan los pesos.

In [ ]:
def cuenta_pasos(n_muestras, tam_lote, epocas):
    pasos_por_epoca = int(np.ceil(n_muestras / tam_lote))
    ultimo = n_muestras % tam_lote or tam_lote
    return pasos_por_epoca, pasos_por_epoca * epocas, ultimo

print(f"{'muestras':>9} {'lote':>6} {'epocas':>7} {'pasos/epoca':>12} "
      f"{'pasos':>8} {'ultimo lote':>12}")
print("-" * 62)
for n, lote, ep in [(300, 32, 100), (300, 300, 100), (60000, 32, 3),
                    (60000, 128, 3), (60000, 512, 3)]:
    ppe, total, ult = cuenta_pasos(n, lote, ep)
    print(f"{n:>9,} {lote:>6} {ep:>7} {ppe:>12} {total:>8,} {ult:>12}")

Mira la segunda fila: con el lote igual a todo el conjunto, cien épocas son **cien**
actualizaciones de los pesos, no diez mil. Eso es el descenso de gradiente por lotes
completos del bloque 5.2, y explica por qué no se usa.

Y mira las tres últimas: con el mismo número de épocas, cuadruplicar el tamaño del lote
deja el entrenamiento con la cuarta parte de los pasos. **Por eso al subir el tamaño de
lote hay que subir la tasa de aprendizaje**, o entrenar más épocas.

---

## 4. `float32`, y cuánto cuesta no hacerlo

Tres mediciones: memoria, velocidad, y qué hace Keras si le pasas `float64`.

In [ ]:
n = 2_000_000
a64 = np.random.default_rng(0).random(n)
a32 = a64.astype("float32")

print(f"float64: {a64.nbytes / 1e6:6.1f} MB")
print(f"float32: {a32.nbytes / 1e6:6.1f} MB   ({a64.nbytes / a32.nbytes:.0f}x menos)")

In [ ]:
import time

def cronometra(f, repeticiones=5):
    f()                                       # una vez para calentar
    t0 = time.perf_counter()
    for _ in range(repeticiones):
        f()
    return (time.perf_counter() - t0) / repeticiones

m64 = np.random.default_rng(0).random((1200, 1200))
m32 = m64.astype("float32")

t64 = cronometra(lambda: m64 @ m64)
t32 = cronometra(lambda: m32 @ m32)

print(f"producto de matrices 1200x1200")
print(f"  float64  {t64 * 1000:7.1f} ms")
print(f"  float32  {t32 * 1000:7.1f} ms   ({t64 / t32:.1f}x mas rapido)")

> **El factor exacto depende de la máquina.** Lo que no depende de la máquina es la
> dirección: `float32` ocupa la mitad y va más rápido, y la precisión extra no aporta nada
> cuando el gradiente ya es una aproximación ruidosa calculada sobre un lote aleatorio.
> Construir `X` en `float32` es gratis y se hace siempre.

---

## 5. Una capa densa es un producto de matrices y una suma

Y ahora la comprobación que cierra el bloque 2: escribir la capa a mano y verificar que
Keras hace exactamente lo mismo.

In [ ]:
def capa_densa(X, W, b, f):
    # Una capa densa, entera. No hay nada mas.
    return f(X @ W + b)


relu = lambda z: np.maximum(0.0, z)

N, n_entradas, n_unidades = 5, 4, 3
rng = np.random.default_rng(20262027)

X = rng.normal(size=(N, n_entradas)).astype("float32")
W = rng.normal(size=(n_entradas, n_unidades)).astype("float32")
b = rng.normal(size=n_unidades).astype("float32")

salida = capa_densa(X, W, b, relu)
print(f"X {X.shape} @ W {W.shape} + b {b.shape}  ->  {salida.shape}")
print()
print(salida.round(3))

In [ ]:
# Lo mismo, con Keras, poniendole los MISMOS pesos.
capa = keras.layers.Dense(n_unidades, activation="relu")
capa.build((None, n_entradas))
capa.set_weights([W, b])

salida_keras = capa(X).numpy()

print("A mano:")
print(salida.round(4))
print()
print("Keras:")
print(salida_keras.round(4))
print()
print("Diferencia maxima:", np.abs(salida - salida_keras).max())
assert np.allclose(salida, salida_keras, atol=1e-5)
print("Identicas. Una capa densa es X @ W + b, y una activacion.")

### El sesgo se suma por difusión

`b` tiene tamaño `(3,)` y se le suma a una matriz `(5, 3)`. NumPy lo estira a las cinco
filas sin copiarlo. Es la difusión de la UD3, y es la razón de que **haya un sesgo por
unidad, no uno por muestra**.

In [ ]:
Z = X @ W
print("X @ W:", Z.shape, "  b:", b.shape, "  ->  Z + b:", (Z + b).shape)
print()
print("Todas las filas reciben el mismo b:")
print((Z + b - Z).round(4))

---

## 6. Contar parámetros a mano

$$\text{parámetros} = m \times (n + 1)$$

con $n$ entradas y $m$ unidades. Hay que saber hacer esta cuenta sin ejecutar nada, y
después comprobarla con `summary()`.

In [ ]:
def parametros_densa(n_entradas, n_unidades):
    return n_unidades * (n_entradas + 1)


# La red de referencia de la unidad sobre TechStore: 10 caracteristicas.
capas = [(10, 16), (16, 8), (8, 1)]
total = 0
print(f"{'capa':>6} {'entradas':>9} {'unidades':>9} {'pesos':>8} {'sesgos':>7} {'total':>7}")
print("-" * 52)
for i, (n, m) in enumerate(capas, 1):
    p = parametros_densa(n, m)
    total += p
    print(f"{i:>6} {n:>9} {m:>9} {n * m:>8} {m:>7} {p:>7}")
print("-" * 52)
print(f"{'':>34} {'TOTAL':>13} {total:>7}")

In [ ]:
modelo = keras.Sequential([
    keras.layers.Input(shape=(10,)),
    keras.layers.Dense(16, activation="relu", name="densa_1"),
    keras.layers.Dense(8, activation="relu", name="densa_2"),
    keras.layers.Dense(1, activation="sigmoid", name="salida"),
])
modelo.summary()

Coincide: **321**.

Y ahora la cuenta que explica el bloque 13 entero. Una imagen en color de 224×224 son
150.528 números. Conectarla a una capa densa de 1.000 unidades cuesta:

In [ ]:
entradas_imagen = 224 * 224 * 3
p = parametros_densa(entradas_imagen, 1000)

print(f"Entradas de una imagen 224x224x3: {entradas_imagen:,}")
print(f"Parametros de UNA capa densa de 1000 unidades: {p:,}")
print(f"En float32, solo los pesos ocupan: {p * 4 / 1e6:.0f} MB")
print()
print("Una capa convolucional de 32 filtros de 3x3 sobre la misma imagen:")
print(f"  {32 * (3 * 3 * 3 + 1):,} parametros, y NO dependen del tamaño de la imagen.")
print()
print(f"Razon: {p / (32 * (3 * 3 * 3 + 1)):,.0f} a 1.")

Ciento sesenta y ocho mil a uno. Ese número es el bloque 13 de los apuntes, y es el motivo
de que las redes convolucionales existan.

---

## Ejercicios

### Ejercicio 1. Formas

Sin ejecutar nada, escribe la forma de:

1. Un lote de 64 imágenes en color de 96×96.
2. La salida de `Dense(10)` aplicada a un tensor `(32, 128)`.
3. Un lote de 16 espectrogramas de 128 frecuencias por 300 instantes, en un canal.
4. El resultado de `np.zeros((5, 3))[..., np.newaxis]`.

Después compruébalas.

### Ejercicio 2. Parámetros

Calcula a mano los parámetros de esta red, capa por capa, y comprueba con `summary()`:

```python
keras.Sequential([
    keras.layers.Input(shape=(20,)),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(3, activation="softmax"),
])
```

¿Qué capa tiene más parámetros, y por qué no es la que tiene más unidades?

### Ejercicio 3. El gradiente a mano

Usa `tf.GradientTape` para calcular la derivada de $f(x) = x^3 - 2x$ en $x = 2$.
Compárala con la derivada analítica, que es $3x^2 - 2$.

Después hazlo con `x = tf.constant(2.0)` en lugar de `tf.Variable`. Falla: devuelve
`None`. Averigua por qué y arréglalo con `cinta.watch(x)`. Explica en una celda de texto
qué está pasando y por qué el comportamiento por defecto tiene sentido.

### Ejercicio 4. El coste de la precisión

Repite la medición del apartado 4 con matrices de 200×200, 600×600 y 2000×2000. ¿La
ventaja de `float32` es la misma en los tres tamaños? Dibuja el resultado con una figura
que pase la lista de comprobación de las once preguntas de la UD4.

### Ejercicio 5. Difusión al revés

¿Qué pasa si a `X @ W`, de forma `(5, 3)`, le sumas un vector de tamaño `(5,)` en lugar de
`(3,)`? Pruébalo. Explica el error, y qué habría que hacer para sumar un valor por muestra
en lugar de por unidad.

---

## Lo que hay que llevarse de aquí

1. Un tensor es un `ndarray` que puede vivir en la GPU y que **recuerda las operaciones
   que lo produjeron**, y eso segundo es lo que permite entrenar.
2. **El eje 0 cuenta muestras**, y una muestra suelta hay que convertirla en un lote de
   una con `x[np.newaxis]`.
3. Lo que importa no son las épocas: son **los pasos**, que son `épocas × muestras / lote`.
4. **`float32` siempre**: la mitad de memoria, más velocidad, y ninguna pérdida útil.
5. **Una capa densa es `X @ W + b` y una activación.** Comprobado contra Keras.
6. Una capa densa tiene $m(n+1)$ parámetros, y sobre una imagen de tamaño real eso son
   ciento cincuenta millones en una sola capa.